In [1]:
import datetime
import os

import disturbancemonitor as dm
from sentinelhub import SHConfig

/home/jviehweger/Documents/Projects/2024/change-detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = SHConfig()

os.environ["SH_CLIENT_ID"] = config.sh_client_id
os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret

input_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [-96.643464, 40.892987],
            [-96.633766, 40.892987],
            [-96.633766, 40.900708],
            [-96.643464, 40.900708],
            [-96.643464, 40.892987],
        ]
    ],
}

monitor = dm.start_monitor(
    name="async-test",
    monitoring_start=datetime.date(2021, 2, 1),
    geometry=input_geojson,
    resolution=0.005,
    backend="AsyncAPI",
    rollback=False,
)

0/6 Initializing model
1/6 Creating bucket
2/6 Fitting model
... Waiting for async request to finish
... Finished
3/6 Writing model to bucket
4/6 Ingesting model to SH
... Waiting for collection to finish ingestion
... Ingested
5/6 Computing metric
... Waiting for async request to finish
Exception occurred: Async request failed: Something went wrong!. 
Rolling back resources.


RuntimeError: Async request failed: Something went wrong!

In [ ]:
try:
    

In [3]:
monitor.async_id

'7c5e9c4d-235a-41d1-8cf2-5281ee43670e'

In [3]:
import boto3

In [5]:
test_session = boto3.session.Session(profile_name="default")

In [9]:
credentials = test_session.get_credentials()

In [11]:
credentials.secret_key

'ctx0EZyEcCy4hXRMObBSGjUpGSgNbwTn3a6rQFWH'

In [3]:
monitor.delete()

In [2]:
config = SHConfig()

os.environ["SH_CLIENT_ID"] = config.sh_client_id
os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret
monitor = dm.load_monitor("async-test", backend="AsyncAPI")

In [3]:
monitor.async_id = "7c5e9c4d-235a-41d1-8cf2-5281ee43670e"
async_id = "7c5e9c4d-235a-41d1-8cf2-5281ee43670e"

In [4]:
from io import BytesIO

import numpy as np
import rasterio
from rasterio.io import MemoryFile

In [9]:
f"s3://{monitor.bucket_name}/{monitor.folder_name}/{monitor.async_id}/metric.tif"

's3://async-test-dcwd8cae/async-test-dcwd8cae/7c5e9c4d-235a-41d1-8cf2-5281ee43670e/default.tif'

In [13]:
with rasterio.open(
    f"s3://{monitor.bucket_name}/{monitor.folder_name}/{monitor.async_id}/metric.tif", opener=monitor.s3.s3_out.open
) as src:
    print(src.overviews(1))

[]


In [6]:
def write_binary(filename, binary):
    with monitor.s3.s3_out.open(filename, "wb") as f:
        f.write(binary.getvalue())


async_out = f"s3://{monitor.bucket_name}/{monitor.folder_name}/{monitor.async_id}/metric.tif"
with rasterio.open(async_out, profile="default") as src:
    profile = src.profile
    profile.update(driver="COG", compress="DEFLATE", blockxsize=1024, blockysize=1024, tiled=True)
    with BytesIO() as out:
        # Create a new file to write to
        with rasterio.open(out, "w", **profile) as dst:
            data = src.read()
            dst.write(src.read())

        with monitor.s3.s3_out.open(f"s3://{monitor.bucket_name}/{monitor.folder_name}/c.tif", "wb") as f:
            f.write(out.getvalue())

    profile.update(count=1)

    zero_raster = np.zeros((profile["height"], profile["width"]), dtype=np.float32)

    with BytesIO() as out:
        with rasterio.open(out, "w", **profile) as dst:
            dst.write(zero_raster, 1)

        with monitor.s3.s3_out.open(f"s3://{monitor.bucket_name}/{monitor.folder_name}/metric.tif", "wb") as f:
            f.write(out.getvalue())

        with monitor.s3.s3_out.open(f"s3://{monitor.bucket_name}/{monitor.folder_name}/disturbedDate.tif", "wb") as f:
            f.write(out.getvalue())

        with monitor.s3.s3_out.open(f"s3://{monitor.bucket_name}/{monitor.folder_name}/process.tif", "wb") as f:
            f.write(out.getvalue())

monitor.s3.s3_out.delete(f"s3://{monitor.bucket_name}/{monitor.folder_name}/{monitor.async_id}", recursive=True)

byoc_url = "https://services.sentinel-hub.com/api/v1/byoc/collections"
new_collection = {"name": monitor.bucket_name, "s3Bucket": monitor.bucket_name}
byoc = monitor.client.post(byoc_url, json=new_collection)
byoc.raise_for_status()
monitor.byoc_id = byoc.json()["data"]["id"]

tile = {
    "path": f"s3://{monitor.bucket_name}/{monitor.folder_name}/(BAND).tif",
    "sensingTime": monitor.monitor_params.monitoring_start,
}
tile_request = monitor.client.post(byoc_url, json=tile)
tile_request.raise_for_status()

In [3]:
process_data = monitor.monitor(end=datetime.date(2021, 4, 1))

In [7]:
monitor.delete()

In [7]:
import datetime
import os

import xarray as xr
from rasterio.io import MemoryFile

with MemoryFile(binary).open() as dataset:
    metrics_ds = xr.open_dataset(dataset, band_as_variable=True, engine="rasterio")
order = ["y", "x"]
reordered_indexes = {index_name: metrics_ds.indexes[index_name] for index_name in order}

In [14]:
reordered_indexes

{'y': Index([40.89877775, 40.89491725], dtype='float64', name='y'),
 'x': Index([-96.64103949999999, -96.6361905], dtype='float64', name='x')}

In [13]:
rename = ["disturbedDate", "process"]
{col: new_name for col, new_name in zip(metrics_ds, rename)}

{'band_1': 'disturbedDate', 'band_2': 'process'}

In [11]:
from dataclasses import dataclass, field


@dataclass
class MonitorParameters:
    name: str
    monitoring_start: str
    geometry: dict
    resolution: tuple
    datasource: str
    harmonics: int = 2
    inputs: list[str] = field(default_factory=lambda: ["NDVI"])
    metric: str = "RMSE"
    last_monitored: datetime.date = None
    sensitivity: int = 5
    boundary: int = 5

    def __post_init__(self):
        if self.last_monitored is None:
            self.last_monitored = self.monitoring_start

In [12]:
kwargs = dict(name="hey", monitoring_start=1, geometry={"b": 1}, resolution=(10, 10), datasource="ay")

test = MonitorParameters(**kwargs)